In [1]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
)

TRAIN_ROOT = (
    DATA_ROOT
    / "BraTS2020_TrainingData"
    / "MICCAI_BraTS2020_TrainingData"
)

VAL_ROOT = (
    DATA_ROOT
    / "BraTS2020_ValidationData"
    / "MICCAI_BraTS2020_ValidationData"
)

# Kiểm tra đường dẫn trước khi đọc dữ liệu.
for name, path in [
    ("DATA_ROOT", DATA_ROOT),
    ("TRAIN_ROOT", TRAIN_ROOT),
    ("VAL_ROOT", VAL_ROOT),
]:
    print(f"{name}: {path}")
    print(f"Exists: {path.is_dir()}\n")

assert DATA_ROOT.is_dir(), f"Không tìm thấy DATA_ROOT: {DATA_ROOT}"
assert TRAIN_ROOT.is_dir(), f"Không tìm thấy TRAIN_ROOT: {TRAIN_ROOT}"
assert VAL_ROOT.is_dir(), f"Không tìm thấy VAL_ROOT: {VAL_ROOT}"

train_subjects = sorted(
    p for p in TRAIN_ROOT.iterdir() if p.is_dir()
)

val_subjects = sorted(
    p for p in VAL_ROOT.iterdir() if p.is_dir()
)

print("Training subjects:", len(train_subjects))
print("Validation subjects:", len(val_subjects))

print("\nFirst training subject:", train_subjects[0].name)
print("First validation subject:", val_subjects[0].name)

DATA_ROOT: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation
Exists: True

TRAIN_ROOT: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
Exists: True

VAL_ROOT: /kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData
Exists: True

Training subjects: 369
Validation subjects: 125

First training subject: BraTS20_Training_001
First validation subject: BraTS20_Validation_001


In [3]:
from pathlib import Path

MODALITIES = ["t1", "t1ce", "t2", "flair"]


def is_nifti_file(path):
    """Kiểm tra file có phần mở rộng NIfTI hay không."""
    name = path.name.lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")


def find_exact_nifti(folder, expected_stem):
    """
    Tìm file có tên chính xác, hỗ trợ .nii và .nii.gz.
    Không tự động chọn file có tên gần giống.
    """
    expected_names = {
        f"{expected_stem}.nii".lower(),
        f"{expected_stem}.nii.gz".lower(),
    }

    matches = [
        p for p in folder.iterdir()
        if p.is_file() and p.name.lower() in expected_names
    ]

    # Nếu không có hoặc có nhiều hơn một kết quả,
    # trả về None để xử lý như trường hợp cần kiểm tra.
    return matches[0] if len(matches) == 1 else None


def relative_path_or_none(path):
    """Lưu đường dẫn tương đối với DATA_ROOT."""
    if path is None:
        return None
    return str(path.relative_to(DATA_ROOT))


def build_manifest(root, split, has_seg):
    records = []

    subject_dirs = sorted(
        p for p in root.iterdir() if p.is_dir()
    )

    for subject_dir in subject_dirs:
        subject_id = subject_dir.name

        row = {
            "subject_id": subject_id,
            "split": split,
        }

        # Bốn modality MRI.
        for modality in MODALITIES:
            expected_stem = f"{subject_id}_{modality}"
            path = find_exact_nifti(
                subject_dir, expected_stem
            )
            row[modality] = relative_path_or_none(path)

        # Chỉ training có mask được kỳ vọng.
        if has_seg:
            seg_path = find_exact_nifti(
                subject_dir, f"{subject_id}_seg"
            )
            row["seg"] = relative_path_or_none(seg_path)

            # Ghi nhận các file có tên chứa "seg".
            # Đây chỉ là danh sách để kiểm tra, không tự gán làm mask.
            candidates = [
                p.name for p in subject_dir.iterdir()
                if p.is_file()
                and is_nifti_file(p)
                and "seg" in p.name.lower()
            ]

            row["seg_candidates"] = "; ".join(candidates)
        else:
            row["seg"] = None
            row["seg_candidates"] = ""

        required_fields = MODALITIES + (
            ["seg"] if has_seg else []
        )

        missing_fields = [
            field for field in required_fields
            if row[field] is None
        ]

        row["missing_files"] = "; ".join(missing_fields)
        row["complete"] = len(missing_fields) == 0

        records.append(row)

    return pd.DataFrame(records)


train_manifest = build_manifest(
    TRAIN_ROOT, split="train", has_seg=True
)

val_manifest = build_manifest(
    VAL_ROOT, split="official_validation", has_seg=False
)

print("Train manifest:", train_manifest.shape)
print("Validation manifest:", val_manifest.shape)

print(
    "Training subjects with all expected files:",
    int(train_manifest["complete"].sum()),
    "/", len(train_manifest)
)

print(
    "Validation subjects with all 4 modalities:",
    int(val_manifest["complete"].sum()),
    "/", len(val_manifest)
)

print("\nIncomplete training subjects:")
display(
    train_manifest.loc[
        ~train_manifest["complete"],
        ["subject_id", "missing_files", "seg_candidates"]
    ]
)

print("\nIncomplete validation subjects:")
display(
    val_manifest.loc[
        ~val_manifest["complete"],
        ["subject_id", "missing_files"]
    ]
)

Train manifest: (369, 10)
Validation manifest: (125, 10)
Training subjects with all expected files: 368 / 369
Validation subjects with all 4 modalities: 125 / 125

Incomplete training subjects:


,subject_id,missing_files,seg_candidates
354,BraTS20_Training_355,seg,W39_1998.09.19_Segm.nii



Incomplete validation subjects:


,subject_id,missing_files


In [4]:
AUDIT_DIR = Path("/kaggle/working/brats_audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

train_manifest.to_csv(
    AUDIT_DIR / "manifest_train.csv",
    index=False
)

val_manifest.to_csv(
    AUDIT_DIR / "manifest_validation.csv",
    index=False
)

print("Saved audit files to:", AUDIT_DIR)

Saved audit files to: /kaggle/working/brats_audit


In [5]:
EXPECTED_SHAPE = (240, 240, 155)
EXPECTED_LABELS = {0, 1, 2, 4}


def audit_manifest(manifest, include_seg, split_name):
    """
    Kiểm tra shape, affine và nhãn.
    Chỉ load dữ liệu segmentation để đọc nhãn;
    không nạp toàn bộ 4 modality vào RAM.
    """
    results = []

    modality_fields = list(MODALITIES)
    required_fields = modality_fields + (
        ["seg"] if include_seg else []
    )

    for i, row in manifest.iterrows():
        subject_id = row["subject_id"]

        missing = [
            field for field in required_fields
            if pd.isna(row[field]) or not str(row[field]).strip()
        ]

        # Nếu thiếu file, ghi nhận và không tiếp tục load subject này.
        if missing:
            results.append({
                "subject_id": subject_id,
                "split": split_name,
                "shape": None,
                "shape_mismatch": "not_checked",
                "affine_mismatch": "not_checked",
                "seg_labels": None,
                "status": "missing_files: " + ",".join(missing),
            })
            continue

        images = {}

        for field in required_fields:
            path = DATA_ROOT / str(row[field])
            images[field] = nib.load(str(path))

        # Lấy FLAIR làm mốc so sánh trong cùng subject.
        reference = images["flair"]
        reference_shape = tuple(reference.shape)
        reference_affine = reference.affine

        shapes = {
            field: tuple(img.shape)
            for field, img in images.items()
        }

        shape_mismatch = [
            field for field, shape in shapes.items()
            if shape != reference_shape
        ]

        affine_mismatch = [
            field for field, img in images.items()
            if not np.allclose(
                img.affine,
                reference_affine,
                rtol=0,
                atol=1e-4,
            )
        ]

        issues = []

        if shape_mismatch:
            issues.append(
                "shape_mismatch:" + ",".join(shape_mismatch)
            )

        if affine_mismatch:
            issues.append(
                "affine_mismatch:" + ",".join(affine_mismatch)
            )

        if reference_shape != EXPECTED_SHAPE:
            issues.append("nonstandard_shape")

        seg_labels_text = None

        if include_seg:
            # Đọc từng mask để giới hạn việc sử dụng bộ nhớ.
            seg_data = np.asanyarray(
                images["seg"].dataobj
            )

            finite_values = bool(np.isfinite(seg_data).all())

            unique_labels = np.unique(seg_data)

            seg_labels_text = ",".join(
                f"{float(value):g}"
                for value in unique_labels
            )

            unexpected_labels = [
                float(value)
                for value in unique_labels
                if (
                    not np.isfinite(value)
                    or value not in EXPECTED_LABELS
                )
            ]

            integer_labels = bool(
                finite_values
                and np.all(seg_data == np.rint(seg_data))
            )

            if not finite_values:
                issues.append("nonfinite_seg_values")

            if not integer_labels:
                issues.append("noninteger_seg_labels")

            if unexpected_labels:
                issues.append(
                    "unexpected_labels:"
                    + ",".join(map(str, unexpected_labels))
                )

        results.append({
            "subject_id": subject_id,
            "split": split_name,
            "shape": str(reference_shape),
            "shape_mismatch": ",".join(shape_mismatch),
            "affine_mismatch": ",".join(affine_mismatch),
            "seg_labels": seg_labels_text,
            "status": "OK" if not issues else "; ".join(issues),
        })

        if (i + 1) % 50 == 0:
            print(
                f"{split_name}: checked {i + 1}/{len(manifest)}"
            )

    return pd.DataFrame(results)


train_audit = audit_manifest(
    train_manifest,
    include_seg=True,
    split_name="train"
)

val_audit = audit_manifest(
    val_manifest,
    include_seg=False,
    split_name="official_validation"
)

train_audit.to_csv(
    AUDIT_DIR / "audit_train.csv",
    index=False
)

val_audit.to_csv(
    AUDIT_DIR / "audit_validation.csv",
    index=False
)

print("\nTraining audit status:")
display(
    train_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="subjects")
)

print("\nValidation audit status:")
display(
    val_audit["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="subjects")
)

print("\nTraining segmentation label patterns:")
display(
    train_audit["seg_labels"]
    .value_counts(dropna=False)
    .rename_axis("labels")
    .reset_index(name="subjects")
)

print("\nTraining subjects requiring review:")
display(
    train_audit.loc[
        train_audit["status"] != "OK"
    ]
)

print("\nValidation subjects requiring review:")
display(
    val_audit.loc[
        val_audit["status"] != "OK"
    ]
)

train: checked 50/369
train: checked 100/369
train: checked 150/369
train: checked 200/369
train: checked 250/369
train: checked 300/369
train: checked 350/369
official_validation: checked 50/125
official_validation: checked 100/125

Training audit status:


,status,subjects
0,OK,368
1,missing_files: seg,1



Validation audit status:


,status,subjects
0,OK,125



Training segmentation label patterns:


,labels,subjects
0,"0,1,2,4",339
1,"0,1,2",27
2,"0,2,4",1
3,"0,1,4",1
4,None,1



Training subjects requiring review:


,subject_id,split,shape,shape_mismatch,affine_mismatch,seg_labels,status
354,BraTS20_Training_355,train,None,not_checked,not_checked,None,missing_files: seg



Validation subjects requiring review:


,subject_id,split,shape,shape_mismatch,affine_mismatch,seg_labels,status


In [6]:
# GIAI ĐOẠN 2: CHIA TẬP THEO BỆNH NHÂN

SEED = 42

# Chỉ lấy các bệnh nhân đã có đầy đủ file cần thiết.
valid_train = train_manifest.loc[
    train_manifest["complete"]
].copy()

# Kiểm tra danh sách đầu vào.
assert valid_train["subject_id"].is_unique, (
    "Có subject_id bị trùng trong manifest."
)

assert valid_train["seg"].notna().all(), (
    "Có bệnh nhân chưa có segmentation mask theo tên chuẩn."
)

n_subjects = len(valid_train)

print("Eligible training subjects:", n_subjects)
print(
    "Excluded subjects:",
    len(train_manifest) - n_subjects
)

# Sắp xếp trước để kết quả không phụ thuộc thứ tự file hệ thống.
subject_ids = np.array(
    sorted(valid_train["subject_id"].tolist())
)

# Xáo trộn có kiểm soát bằng seed.
rng = np.random.default_rng(SEED)
rng.shuffle(subject_ids)

# Chia theo bệnh nhân, không chia theo slice.
n_train = int(n_subjects * 0.70)
n_val = int(n_subjects * 0.15)
n_test = n_subjects - n_train - n_val

train_ids = subject_ids[:n_train]
val_ids = subject_ids[n_train:n_train + n_val]
test_ids = subject_ids[n_train + n_val:]

print("\nSplit sizes:")
print("Train:", len(train_ids))
print("Internal validation:", len(val_ids))
print("Test:", len(test_ids))

Eligible training subjects: 368
Excluded subjects: 1

Split sizes:
Train: 257
Internal validation: 55
Test: 56


In [7]:
# CHUYỂN DANH SÁCH ID THÀNH SET ĐỂ KIỂM TRA GIAO NHAU

train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

assert train_set.isdisjoint(val_set), (
    "Lỗi: train và validation có bệnh nhân giao nhau."
)

assert train_set.isdisjoint(test_set), (
    "Lỗi: train và test có bệnh nhân giao nhau."
)

assert val_set.isdisjoint(test_set), (
    "Lỗi: validation và test có bệnh nhân giao nhau."
)

# Tổng số bệnh nhân phải được giữ nguyên.
all_split_ids = train_set | val_set | test_set

assert len(all_split_ids) == n_subjects, (
    "Lỗi: số lượng ID sau khi chia không khớp."
)

print("PASS: Không có bệnh nhân bị trùng giữa ba tập.")
print("PASS: Mỗi bệnh nhân hợp lệ được phân vào đúng một tập.")

PASS: Không có bệnh nhân bị trùng giữa ba tập.
PASS: Mỗi bệnh nhân hợp lệ được phân vào đúng một tập.


In [8]:
# TẠO BẢNG PHÂN CHIA

split_assignments = pd.concat(
    [
        pd.DataFrame({
            "subject_id": train_ids,
            "split": "train",
        }),
        pd.DataFrame({
            "subject_id": val_ids,
            "split": "internal_validation",
        }),
        pd.DataFrame({
            "subject_id": test_ids,
            "split": "test",
        }),
    ],
    ignore_index=True,
)

split_assignments["seed"] = SEED

# Kiểm tra mỗi subject xuất hiện đúng một lần.
assert split_assignments["subject_id"].is_unique

# Bảng số lượng và tỷ lệ từng tập.
split_summary = (
    split_assignments["split"]
    .value_counts()
    .reindex(["train", "internal_validation", "test"])
    .rename_axis("split")
    .to_frame("n_subjects")
)

split_summary["percentage"] = (
    split_summary["n_subjects"] / n_subjects * 100
)

display(split_summary)

# Lưu danh sách những bệnh nhân chưa được đưa vào split.
excluded_subjects = train_manifest.loc[
    ~train_manifest["complete"]
].copy()

# Lưu vào thư mục output của Kaggle.
AUDIT_DIR = Path("/kaggle/working/brats_audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

split_assignments.to_csv(
    AUDIT_DIR / "split_assignments_seed42.csv",
    index=False,
)

excluded_subjects.to_csv(
    AUDIT_DIR / "excluded_subjects.csv",
    index=False,
)

# Ghép thông tin split với manifest.
# Bỏ cột split cũ vì train_manifest có split="train" từ lúc tạo.
split_manifest = (
    valid_train
    .drop(columns=["split"], errors="ignore")
    .merge(
        split_assignments,
        on="subject_id",
        how="inner",
        validate="one_to_one",
    )
)

assert len(split_manifest) == n_subjects

# Lưu manifest riêng cho từng tập.
for split_name in [
    "train",
    "internal_validation",
    "test",
]:
    subset = split_manifest.loc[
        split_manifest["split"] == split_name
    ].copy()

    subset.to_csv(
        AUDIT_DIR / f"manifest_{split_name}.csv",
        index=False,
    )

print("\nSaved split files to:", AUDIT_DIR)
print("Split and manifest creation completed.")

,n_subjects,percentage
split,,
train,257,69.836957
internal_validation,55,14.945652
test,56,15.217391



Saved split files to: /kaggle/working/brats_audit
Split and manifest creation completed.


In [9]:
# GIAI ĐOẠN 3 — CELL 1: KIỂM TRA MANIFEST

from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

DATA_ROOT = Path(
    "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
)

assert "split_manifest" in globals(), (
    "Chưa có split_manifest. Hãy chạy lại các cell chia tập trước."
)

required_columns = {
    "subject_id",
    "t1",
    "t1ce",
    "t2",
    "flair",
    "seg",
    "split",
}

assert required_columns.issubset(split_manifest.columns), (
    f"Thiếu cột: {required_columns - set(split_manifest.columns)}"
)

print("Dataset root exists:", DATA_ROOT.is_dir())
print("Number of subjects:", len(split_manifest))

print("\nSubjects by split:")
display(
    split_manifest["split"]
    .value_counts()
    .reindex(["train", "internal_validation", "test"])
    .rename_axis("split")
    .to_frame("n_subjects")
)

assert DATA_ROOT.is_dir()
assert split_manifest["subject_id"].is_unique
assert split_manifest["seg"].notna().all()

print("\nManifest checks passed.")

Dataset root exists: True
Number of subjects: 368

Subjects by split:


,n_subjects
split,
train,257
internal_validation,55
test,56



Manifest checks passed.


In [10]:
# GIAI ĐOẠN 3 — CELL 2: CÁC HÀM TIỀN XỬ LÝ

MODALITIES = ["t1", "t1ce", "t2", "flair"]

ORIGINAL_LABELS = {0, 1, 2, 4}


def normalize_mri(volume, modality_name="unknown"):
    """
    Chuẩn hóa cường độ của một modality MRI.

    Input:
        volume: NumPy array 3D.

    Output:
        NumPy array 3D, dtype float32.

    Các voxel bằng 0 được giữ nguyên là 0.
    """

    volume = np.asarray(volume, dtype=np.float32)

    if volume.ndim != 3:
        raise ValueError(
            f"{modality_name}: expected 3D volume, "
            f"received shape {volume.shape}"
        )

    if not np.isfinite(volume).all():
        raise ValueError(
            f"{modality_name}: found NaN or Inf values"
        )

    foreground = volume != 0

    if not foreground.any():
        raise ValueError(
            f"{modality_name}: the entire volume is zero"
        )

    values = volume[foreground]

    mean = float(values.mean(dtype=np.float64))
    std = float(values.std(dtype=np.float64))

    if not np.isfinite(std) or std < 1e-6:
        raise ValueError(
            f"{modality_name}: invalid or near-zero standard deviation"
        )

    normalized = np.zeros_like(volume, dtype=np.float32)

    normalized[foreground] = (
        volume[foreground] - mean
    ) / std

    return normalized


def remap_brats_labels(segmentation):
    """
    Chuyển nhãn BraTS gốc thành nhãn liên tục:

        0 -> 0: Background
        1 -> 1
        2 -> 2
        4 -> 3

    Output dtype: uint8.
    """

    if not np.isfinite(segmentation).all():
        raise ValueError("Segmentation contains NaN or Inf")

    rounded = np.rint(segmentation)

    if not np.allclose(segmentation, rounded):
        raise ValueError("Segmentation contains non-integer labels")

    rounded = rounded.astype(np.uint8)

    unique_labels = set(
        np.unique(rounded).astype(int).tolist()
    )

    unexpected = unique_labels - ORIGINAL_LABELS

    if unexpected:
        raise ValueError(
            f"Unexpected original labels: {sorted(unexpected)}"
        )

    remapped = np.zeros(rounded.shape, dtype=np.uint8)

    remapped[rounded == 1] = 1
    remapped[rounded == 2] = 2
    remapped[rounded == 4] = 3

    return remapped

In [12]:
# Kiểm tra normalize_mri
test_volume = np.zeros((4, 4, 4), dtype=np.float32)

test_volume[1:3, 1:3, 1:3] = np.arange(
    1, 9, dtype=np.float32
).reshape(2, 2, 2)

test_normalized = normalize_mri(test_volume, "test")

assert test_normalized.shape == test_volume.shape
assert test_normalized.dtype == np.float32
assert np.isfinite(test_normalized).all()
assert np.all(test_normalized[test_volume == 0] == 0)


# Kiểm tra remap nhãn
test_seg = np.array([0, 1, 2, 4], dtype=np.uint8)

test_remapped = remap_brats_labels(test_seg)

assert test_remapped.tolist() == [0, 1, 2, 3]


print("Normalization test passed.")
print("Label remapping test passed.")

Normalization test passed.
Label remapping test passed.


In [26]:
# GIAI ĐOẠN 3 — CELL 3: DATASET CLASS

class BraTSVolumeDataset(Dataset):
    def __init__(self, manifest, data_root):
        self.manifest = manifest.reset_index(drop=True).copy()
        self.data_root = Path(data_root)

        required = MODALITIES + ["seg", "subject_id"]

        missing_columns = set(required) - set(
            self.manifest.columns
        )

        if missing_columns:
            raise ValueError(
                f"Manifest is missing columns: {missing_columns}"
            )

        # Kiểm tra file tồn tại mà chưa load volume vào RAM.
        for _, row in self.manifest.iterrows():
            for field in MODALITIES + ["seg"]:
                rel_path = row[field]

                if pd.isna(rel_path) or not str(rel_path).strip():
                    raise ValueError(
                        f"{row['subject_id']} is missing {field}"
                    )

                path = self.data_root / str(rel_path)

                if not path.is_file():
                    raise FileNotFoundError(
                        f"File not found: {path}"
                    )

    def __len__(self):
        return len(self.manifest)

    def __getitem__(self, index):
        row = self.manifest.iloc[index]
        subject_id = row["subject_id"]

        # 1. Load bốn modality.
        nifti_images = {}

        for modality in MODALITIES:
            path = self.data_root / str(row[modality])
            nifti_images[modality] = nib.load(str(path))

        # Dùng FLAIR làm mốc so sánh hình học.
        reference = nifti_images["flair"]
        reference_shape = tuple(reference.shape)
        reference_affine = reference.affine

        # 2. Kiểm tra shape và affine tại thời điểm đọc.
        for modality, nii_img in nifti_images.items():
            if tuple(nii_img.shape) != reference_shape:
                raise ValueError(
                    f"{subject_id}: shape mismatch in {modality}"
                )

            if not np.allclose(
                nii_img.affine,
                reference_affine,
                rtol=0,
                atol=1e-4,
            ):
                raise ValueError(
                    f"{subject_id}: affine mismatch in {modality}"
                )

        # 3. Normalize và giữ đúng thứ tự kênh.
        channels = []

        for modality in MODALITIES:
            volume = nifti_images[modality].get_fdata(
                dtype=np.float32,
                caching="unchanged",
            )

            normalized = normalize_mri(
                volume,
                modality_name=f"{subject_id}_{modality}",
            )

            channels.append(normalized)

        # Shape: [4, D, H, W]
        image_array = np.stack(channels, axis=0)

        # 4. Load và kiểm tra segmentation mask.
        seg_path = self.data_root / str(row["seg"])
        seg_img = nib.load(str(seg_path))

        if tuple(seg_img.shape) != reference_shape:
            raise ValueError(
                f"{subject_id}: segmentation shape mismatch"
            )

        if not np.allclose(
            seg_img.affine,
            reference_affine,
            rtol=0,
            atol=1e-4,
        ):
            raise ValueError(
                f"{subject_id}: segmentation affine mismatch"
            )

        seg_raw = seg_img.get_fdata(
            dtype=np.float32,
            caching="unchanged",
        )

        label_array = remap_brats_labels(seg_raw)

        # 5. Chuyển thành PyTorch tensors.
        image_tensor = torch.from_numpy(
            np.ascontiguousarray(image_array, dtype=np.float32)
        )

        # Giữ target dạng uint8 để tiết kiệm bộ nhớ.
        # Khi tính CrossEntropyLoss, ta sẽ chuyển sang torch.long.
        label_tensor = torch.from_numpy(
            np.ascontiguousarray(label_array, dtype=np.uint8)
        )

        return {
            "image": image_tensor,
            "label": label_tensor,
            "subject_id": subject_id,
        }

In [27]:
# GIAI ĐOẠN 3 — CELL 4: TẠO DATASET VÀ DATALOADER

train_df = split_manifest.loc[
    split_manifest["split"] == "train"
].copy()

val_df = split_manifest.loc[
    split_manifest["split"] == "internal_validation"
].copy()

test_df = split_manifest.loc[
    split_manifest["split"] == "test"
].copy()

train_dataset = BraTSVolumeDataset(train_df, DATA_ROOT)
val_dataset = BraTSVolumeDataset(val_df, DATA_ROOT)
test_dataset = BraTSVolumeDataset(test_df, DATA_ROOT)

loader_generator = torch.Generator()
loader_generator.manual_seed(42)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    generator=loader_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Dataset sizes:")
print("Train:", len(train_dataset))
print("Internal validation:", len(val_dataset))
print("Test:", len(test_dataset))

Dataset sizes:
Train: 257
Internal validation: 55
Test: 56


In [28]:
# GIAI ĐOẠN 3 — CELL 5: KIỂM TRA BATCH ĐẦU TIÊN

batch = next(iter(train_loader))

images = batch["image"]
labels = batch["label"]
subject_ids = batch["subject_id"]

print("Subject:", subject_ids[0])
print("Image shape:", tuple(images.shape))
print("Image dtype:", images.dtype)
print("Label shape:", tuple(labels.shape))
print("Label dtype:", labels.dtype)

print("\nImage values:")
print("Min:", images.min().item())
print("Max:", images.max().item())

print("\nUnique remapped labels:")
print(torch.unique(labels).tolist())

# Các kiểm tra bắt buộc.
assert images.ndim == 5
assert images.shape[0] == 1
assert images.shape[1] == 4

assert labels.ndim == 4
assert tuple(labels.shape[1:]) == tuple(images.shape[2:])

assert images.dtype == torch.float32
assert labels.dtype == torch.uint8

assert torch.isfinite(images).all()
assert set(torch.unique(labels).tolist()).issubset({0, 1, 2, 3})

print("\nPASS: DataLoader returned a valid batch.")

Subject: BraTS20_Training_286
Image shape: (1, 4, 240, 240, 155)
Image dtype: torch.float32
Label shape: (1, 240, 240, 155)
Label dtype: torch.uint8

Image values:
Min: -3.2895989418029785
Max: 10.294367790222168

Unique remapped labels:
[0, 1, 2]

PASS: DataLoader returned a valid batch.
